In [0]:
from pyspark.sql.functions import from_json, col, coalesce, current_timestamp, unbase64, expr
from pyspark.sql.types import *

schema = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("id", IntegerType()),
            StructField("product", StringType()),
            StructField("price", StructType([
                StructField("scale", IntegerType()),
                StructField("value", StringType())
            ])),
            StructField("created_at", LongType())
        ])),
        StructField("after", StructType([
            StructField("id", IntegerType()),
            StructField("product", StringType()),
            StructField("price", StructType([
                StructField("scale", IntegerType()),
                StructField("value", StringType())
            ])),
            StructField("created_at", LongType())
        ])),
        StructField("op", StringType()),
        StructField("ts_ms", LongType())
    ]))
])

In [0]:

# bronze = spark.read.format("delta").load(
# "/Volumes/workspace/default/mnt/bronze/orders"
# )

# cdc = bronze.select(
#     from_json(col("value"), schema).alias("data")
# ).select("data.payload.*")

bronze_stream = (
    spark.readStream
    .format("delta")
    .load("/Volumes/workspace/default/mnt/bronze/orders")
)

parsed = bronze_stream.select(
    from_json(col("value"), schema).alias("data")
).select("data.payload.*")


In [0]:
events = parsed.select(
    coalesce(col("after.id"), col("before.id")).alias("id"),
    coalesce(col("after.product"), col("before.product")).alias("product"),

    col("after.price.value").alias("price_raw"),
    col("after.price.scale").alias("price_scale"),

    col("after.created_at").alias("created_at_raw"),

    col("op"),

    current_timestamp().alias("event_time")
)

In [0]:
events = events.withColumn(
    "price_bytes",
    unbase64(col("price_raw"))
)

events = events.withColumn(
    "price_int",
    expr("conv(hex(price_bytes),16,10)").cast("double")
)

events = events.withColumn(
    "price",
    col("price_int") / pow(10, col("price_scale"))
    
)

events = events.withColumn(
    "created_at",
    (col("created_at_raw") / 1000000).cast("timestamp")
)

In [0]:
events = events.select(
    "id",
    "product",
    "price",
    "created_at",
    "op",
    "event_time"
)

In [0]:
def upsert_to_silver(batch_df, batch_id):


    from delta.tables import DeltaTable
    from pyspark.sql.functions import col, coalesce, current_timestamp, row_number
    from pyspark.sql.window import Window

    if batch_df.count() == 0:
        return

    w = Window.partitionBy("id").orderBy(col("event_time").desc())

    batch_df = (
        batch_df
        .withColumn("rn", row_number().over(w))
        .filter(col("rn") == 1)
        .drop("rn")
    )
    delta_table = DeltaTable.forName(spark,"silver.silver_orders")

    (
        delta_table.alias("t")
        .merge(batch_df.alias("s"), "t.id = s.id")

        .whenMatchedUpdate(
            condition="s.op != 'd'",
            set={
                "product":"s.product",
                "price":"s.price",
                "created_at":"s.created_at",
                "last_updated_dt":"s.event_time"
            }
        )

        .whenMatchedDelete(
            condition="s.op='d'"
        )

        .whenNotMatchedInsert(
            values={
                "id":"s.id",
                "product":"s.product",
                "price":"s.price",
                "created_at":"s.created_at",
                "last_inserted_dt":"s.event_time",
                "last_updated_dt":"s.event_time"
            }
        )
        .execute()
    )

In [0]:

(
events.writeStream
.foreachBatch(upsert_to_silver)
.option(
  "checkpointLocation",
  "/Volumes/workspace/default/mnt/checkpoints/orders_silver"
)
.trigger(availableNow=True)
.start()
)
